In [1]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [2]:
import kagglehub

# Download latest version
path = kagglehub.model_download("google/paligemma-2/transformers/paligemma2-3b-pt-224")

print("Path to model files:", path)

Path to model files: /kaggle/input/paligemma-2/transformers/paligemma2-3b-pt-224/1


In [3]:
# Install dependencies
!pip install -q transformers peft nltk rouge-score wandb

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 29.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 76.2 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcugraph-cu12 24.12.0 requires pylibraft-cu12==24.12.*, but you have pylibraft-cu12 25.2.0 which is incompatible.
pylibcugraph-cu12 24.12.0 

In [4]:
# Verify GPU
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.5.1+cu124
True


In [5]:
import wandb
wandb.login(key="d070aabfe54f4733fb727662604b037dee34842c")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adigew (adigew-middle-east-technical-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [6]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory

data = pd.read_csv("/kaggle/input/riscmm/RISCM/captions.csv")
data.head()

,source,split,image,caption_1,caption_2,caption_3,caption_4,caption_5
0,NWPU,test,NWPU_31430.jpg,A gray plane on the runway and the lawn beside .,A grey plane is on the runway by the lawn .,There is an airplane on the runway with a larg...,A plane is parked on the runway next to the gr...,There is a plane on the runway beside the grass .
1,NWPU,test,NWPU_31431.jpg,Three small planes parked in a line on the air...,"There are four aircraft on the open ground, Th...",There are many planes of different sizes in a ...,Four planes are parked on the runway .,Four planes of different sizes were on the mar...
2,NWPU,test,NWPU_31432.jpg,A plane parked in a line on the airport with s...,A white plane was parked on the instruction li...,An airplane parked in an open area with many c...,A plane is parked on the open space .,There is 1 plane on the ground marked .
3,NWPU,test,NWPU_31433.jpg,A small plane and a big plane parked next to b...,A white plane and a gray plane parked at the b...,Two planes of different sizes are neatly parke...,A large plane and a small plane are parked nea...,Two planes are on the marked ground .
4,NWPU,test,NWPU_31434.jpg,Two planes parked next to boarding bridges .,Two aircraft were parked at the departure gates .,Two planes of different sizes are neatly parke...,Two planes are parked next to the terminal .,Two planes are on the marked ground .


Load Dataset with Small Partition

In [7]:
import pandas as pd
import os

def load_small_partition(image_dir, caption_file, sample_size=300):
    df = pd.read_csv(caption_file)

    # Filter out missing images
    valid_images = [f for f in os.listdir(image_dir) if os.path.isfile(os.path.join(image_dir, f))]
    df = df[df['image'].isin(valid_images)]

    # Use existing splits
    train_df = df[df['split'] == 'train'].sample(frac=1, random_state=42).head(sample_size)
    val_df = df[df['split'] == 'test'].sample(frac=1, random_state=42).head(int(0.2 * sample_size))

    print(f"Loaded small partition: {len(train_df)} train, {len(val_df)} val")
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)

# Set paths
image_dir = "/kaggle/input/riscmm/RISCM/resized"
caption_file = "/kaggle/input/riscmm/RISCM/captions.csv"

# Load data
train_df, val_df = load_small_partition(image_dir, caption_file, sample_size=8000)

Loaded small partition: 8000 train, 1600 val


In [8]:
# Display the DataFrames
print("\n--- Train DataFrame (first 5 rows) ---")
print(train_df.head())

print("\n--- Validation DataFrame (first 5 rows) ---")
print(val_df.head())


--- Train DataFrame (first 5 rows) ---
  source  split           image  \
0  RSICD  train  RSICD_7381.jpg   
1  RSICD  train  RSICD_3740.jpg   
2   NWPU  train   NWPU_8720.jpg   
3  RSICD  train  RSICD_7720.jpg   
4   NWPU  train   NWPU_2940.jpg   

                                           caption_1  \
0  some green plants and many farmlands are in tw...   
1        many green trees are in a piece of forest .   
2  The forest has a lot of randomly arranged gree...   
3  the stretch of turbid saddle shaped river is a...   
4  A blue bridge built on a dark green river and ...   

                                           caption_2  \
0  many green trees and some buildings are in a r...   
1        many green trees are in a piece of forest .   
2  This is a dense forest, And the trees are dark...   
3  we can see the turning area of a river in the ...   
4     Many ships docked on both sides of the river .   

                                           caption_3  \
0  many green trees

Training Function

In [9]:
from transformers import PaliGemmaForConditionalGeneration

# Load the model
model = PaliGemmaForConditionalGeneration.from_pretrained("/kaggle/input/paligemma-2/transformers/paligemma2-3b-pt-224/1")

# Print all module names that might be usable for LoRA
for name, module in model.named_modules():
    if any(key in name for key in ["proj", "fc", "dense", "mlp", "gate"]):
        print(name)


2025-06-15 09:54:38.144640: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749981278.338151      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749981278.394101      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

vision_tower.vision_model.encoder.layers.0.self_attn.k_proj
vision_tower.vision_model.encoder.layers.0.self_attn.v_proj
vision_tower.vision_model.encoder.layers.0.self_attn.q_proj
vision_tower.vision_model.encoder.layers.0.self_attn.out_proj
vision_tower.vision_model.encoder.layers.0.mlp
vision_tower.vision_model.encoder.layers.0.mlp.activation_fn
vision_tower.vision_model.encoder.layers.0.mlp.fc1
vision_tower.vision_model.encoder.layers.0.mlp.fc2
vision_tower.vision_model.encoder.layers.1.self_attn.k_proj
vision_tower.vision_model.encoder.layers.1.self_attn.v_proj
vision_tower.vision_model.encoder.layers.1.self_attn.q_proj
vision_tower.vision_model.encoder.layers.1.self_attn.out_proj
vision_tower.vision_model.encoder.layers.1.mlp
vision_tower.vision_model.encoder.layers.1.mlp.activation_fn
vision_tower.vision_model.encoder.layers.1.mlp.fc1
vision_tower.vision_model.encoder.layers.1.mlp.fc2
vision_tower.vision_model.encoder.layers.2.self_attn.k_proj
vision_tower.vision_model.encoder.la

In [10]:
import os
import torch
from transformers import PaliGemmaForConditionalGeneration, PaliGemmaProcessor
from peft import LoraConfig, get_peft_model
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import torch.optim as optim
import wandb
import string
import random

# Caption cleaner for consistency with evaluation
def preprocess_caption(caption):
    return caption.lower().translate(str.maketrans("", "", string.punctuation)).strip()

class RISCDataset(Dataset):
    def __init__(self, image_dir, df):
        self.image_dir = image_dir
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.image_dir, row.image)
        image = Image.open(image_path).convert('RGB')
        caption = random.choice([row[f"caption_{i}"] for i in range(1, 6)])
        caption = preprocess_caption(caption)
        return {"image": image, "caption": caption}

def custom_collate_fn(batch):
    images = [item["image"] for item in batch]
    captions = [item["caption"] for item in batch]
    return {"images": images, "captions": captions}

def train_lora(model_name, image_dir, train_df, val_df, caption_file, output_dir,
               lora_rank=32, epochs=2, learning_rate=5e-4,
               max_train_samples=None, max_val_samples=None,
               batch_size=2, accum_steps=8,
               target_modules=["q_proj", "v_proj"]):

    wandb.init(project="DI725_Phase3", name=f"LoRA-R{lora_rank}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = PaliGemmaForConditionalGeneration.from_pretrained(model_name, torch_dtype=torch.float16).to(device)
    processor = PaliGemmaProcessor.from_pretrained(model_name, use_fast=True)

    lora_config = LoraConfig(
        r=lora_rank,
        lora_alpha=32,
        target_modules=target_modules,
        lora_dropout=0.1,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)

    if max_train_samples:
        train_df = train_df.head(max_train_samples)
    if max_val_samples:
        val_df = val_df.head(max_val_samples)

    train_dataset = RISCDataset(image_dir, train_df)
    val_dataset = RISCDataset(image_dir, val_df)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=custom_collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=custom_collate_fn)

    optimizer = optim.AdamW(model.parameters(), lr=learning_rate)
    scaler = GradScaler()

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        steps = 0
        optimizer.zero_grad()

        for batch_idx, batch in enumerate(train_loader):
            try:
                images = batch["images"]
                captions = [f"<image> {cap}" for cap in batch["captions"]]

                inputs = processor(text=captions, images=images, return_tensors="pt", padding="longest").to(device)

                with autocast("cuda"):
                    outputs = model(
                        input_ids=inputs["input_ids"],
                        attention_mask=inputs["attention_mask"],
                        pixel_values=inputs["pixel_values"],
                        labels=inputs["input_ids"]
                    )
                    loss = outputs.loss / accum_steps

                scaler.scale(loss).backward()

                if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == len(train_loader):
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()

                total_loss += loss.item() * accum_steps
                steps += 1

                if steps % 100 == 0:
                    print(f"Epoch {epoch+1}, Step {steps}, Loss: {loss.item() * accum_steps:.4f}")

            except Exception as e:
                print(f"⚠️ Error in batch {batch_idx}: {e}")
                continue

        avg_train_loss = total_loss / steps if steps > 0 else 0
        wandb.log({"epoch": epoch+1, "train_loss": avg_train_loss})

        # Validation
        model.eval()
        val_loss = 0
        val_steps = 0
        for batch in val_loader:
            try:
                images = batch["images"]
                captions = [f"<image> {cap}" for cap in batch["captions"]]
                inputs = processor(text=captions, images=images, return_tensors="pt", padding="longest").to(device)

                with torch.no_grad(), autocast("cuda"):
                    outputs = model(**inputs, labels=inputs["input_ids"])
                    val_loss += outputs.loss.item()
                val_steps += 1

            except Exception as e:
                print(f"⚠️ Validation error: {e}")
                continue

        avg_val_loss = val_loss / val_steps if val_steps > 0 else 0
        wandb.log({"epoch": epoch+1, "val_loss": avg_val_loss})
        print(f"Epoch {epoch+1}, Validation Loss: {avg_val_loss:.4f}")
        model.train()

    # Save model + processor
    model.save_pretrained(output_dir)
    processor.save_pretrained(output_dir)
    wandb.finish()


Evaluation Functions

In [11]:
!pip install evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 7.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.12.0 which is incompatible.
bigframes 1.36.0 requires rich<14,>=12.4.4, but you have rich 14.0.0 which is incompatible.


In [12]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from transformers import PaliGemmaProcessor, PaliGemmaForConditionalGeneration
from peft import PeftModel
from PIL import Image
import torch
from torch.amp import autocast
import pandas as pd
import os
import string
import evaluate

# For better CUDA memory management
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Initialize metrics
meteor_metric = evaluate.load("meteor")
smoothie = SmoothingFunction().method1

def preprocess_caption(caption):
    return caption.lower().translate(str.maketrans("", "", string.punctuation)).strip()

def get_all_references(row):
    return [preprocess_caption(row[f"caption_{j}"]) for j in range(1, 6)]

def evaluate_zero_shot(model_name, image_dir, val_df, num_samples=10):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = PaliGemmaForConditionalGeneration.from_pretrained(model_name).to(device).to(torch.float16)
    processor = PaliGemmaProcessor.from_pretrained(model_name)
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

    bleu_scores, rouge_scores, meteor_scores = [], [], []
    val_df = val_df.reset_index(drop=True)
    print("Running zero-shot evaluation...")

    for i in range(min(num_samples, len(val_df))):
        try:
            row = val_df.iloc[i]
            image_path = os.path.join(image_dir, row.image)
            if not os.path.exists(image_path):
                continue

            image = Image.open(image_path).convert('RGB')
            prompt = "<image> caption"
            inputs = processor(text=prompt, images=image, return_tensors="pt").to(device)

            with torch.inference_mode(), autocast("cuda", dtype=torch.float16):
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=50,
                    num_beams=3,
                    repetition_penalty=1.2,
                    eos_token_id=processor.tokenizer.eos_token_id
                )
                caption = processor.decode(output_ids[0], skip_special_tokens=True).strip()

            hypothesis = preprocess_caption(caption)
            references = get_all_references(row)

            if len(hypothesis.split()) < 3 or "##" in hypothesis or hypothesis.count("caption") > 3:
                print(f"Invalid caption generated at sample {i}: '{hypothesis}'. Skipping...")
                continue

            bleu = sentence_bleu([ref.split() for ref in references], hypothesis.split(), smoothing_function=smoothie)
            rouge = scorer.score(references[0], hypothesis)['rougeL'].fmeasure
            meteor = meteor_metric.compute(predictions=[hypothesis], references=[references])["meteor"]

            bleu_scores.append(bleu)
            rouge_scores.append(rouge)
            meteor_scores.append(meteor)

            print(f"\nSample {i} - Hypothesis: {caption}\nBLEU={bleu:.3f} ROUGE-L={rouge:.3f} METEOR={meteor:.3f}")

        except Exception as e:
            print(f"Error evaluating sample {i}: {e}")
        finally:
            torch.cuda.empty_cache()

    return {
        "zero_shot_BLEU": sum(bleu_scores)/len(bleu_scores),
        "zero_shot_ROUGE_L": sum(rouge_scores)/len(rouge_scores),
        "zero_shot_METEOR": sum(meteor_scores)/len(meteor_scores)
    }

def evaluate_lora_model(lora_model_path, base_model_name, image_dir, val_df, num_samples=10):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    base_model = PaliGemmaForConditionalGeneration.from_pretrained(base_model_name).to(device).to(torch.float16)
    model = PeftModel.from_pretrained(base_model, lora_model_path).to(device)
    processor = PaliGemmaProcessor.from_pretrained(base_model_name)
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

    bleu_scores, rouge_scores, meteor_scores = [], [], []

    prompt_prefixes = [
        "<image> Describe this remote sensing image.",
        "<image> What is shown in this satellite image?",
        "<image> Generate a one-sentence description of this image."
    ]

    print("Running LoRA model evaluation...")
    val_df = val_df.reset_index(drop=True)

    for i in range(min(num_samples, len(val_df))):
        try:
            row = val_df.iloc[i]
            image_path = os.path.join(image_dir, row.image)
            if not os.path.exists(image_path):
                continue

            image = Image.open(image_path).convert('RGB')
            prompt = prompt_prefixes[i % len(prompt_prefixes)]

            inputs = processor(text=prompt, images=image, return_tensors="pt").to(device)

            with torch.inference_mode(), autocast("cuda", dtype=torch.float16):
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=50,
                    num_beams=3,
                    temperature=0.9,
                    top_p=0.9,
                    repetition_penalty=1.1,
                    no_repeat_ngram_size=4,
                    eos_token_id=processor.tokenizer.eos_token_id,
                    pad_token_id=processor.tokenizer.pad_token_id
                )
                decoded = processor.decode(output_ids[0], skip_special_tokens=True).strip()

            # Remove prompt if it’s echoed
            for prefix in prompt_prefixes:
                prompt_text = prefix.replace("<image>", "").strip().lower()
                if decoded.lower().startswith(prompt_text):
                    decoded = decoded[len(prompt_text):].strip()

            hypothesis = preprocess_caption(decoded)
            references = [preprocess_caption(row[f"caption_{j}"]) for j in range(1, 6)]

            if len(hypothesis.split()) < 3 or hypothesis.lower().count("caption") > 3:
                print(f"Invalid caption generated at sample {i}: '{hypothesis}'. Skipping...")
                continue

            bleu = sentence_bleu([ref.split() for ref in references], hypothesis.split(), smoothing_function=smoothie)
            rouge = scorer.score(references[0], hypothesis)['rougeL'].fmeasure
            meteor = meteor_metric.compute(predictions=[hypothesis], references=[references])["meteor"]

            bleu_scores.append(bleu)
            rouge_scores.append(rouge)
            meteor_scores.append(meteor)

            print(f"\nSample {i} - Hypothesis: {decoded}\nBLEU={bleu:.3f} ROUGE-L={rouge:.3f} METEOR={meteor:.3f}")

        except Exception as e:
            print(f"⚠️ Error evaluating sample {i}: {e}")
        finally:
            torch.cuda.empty_cache()

    return {
        "lora_BLEU": sum(bleu_scores)/len(bleu_scores) if bleu_scores else 0.0,
        "lora_ROUGE_L": sum(rouge_scores)/len(rouge_scores) if rouge_scores else 0.0,
        "lora_METEOR": sum(meteor_scores)/len(meteor_scores) if meteor_scores else 0.0
    }


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


Train Multiple LoRA Configurations

In [13]:
from transformers import PaliGemmaForConditionalGeneration, PaliGemmaProcessor
from peft import PeftModel
import wandb
import os
import pandas as pd

# Log in to Weights & Biases
wandb.login()

# Set environment variable for better memory management
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Evaluation sample size
num_samples = 60

# LoRA configurations
#configs = [
#    {"lora_rank": 32, "name": "LoRA-R32"},
    #{"lora_rank": 32, "name": "LoRA-R32-k-o", "target_modules": ["k_proj", "o_proj"]}
#]

configs = [
    #{"lora_rank": 32, "name": "LoRA-qv", "target_modules": ["q_proj", "v_proj"]},
    #{"lora_rank": 32, "name": "LoRA-attn", "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"]},
    {"lora_rank": 16, "name": "LoRA-attn-mlp", "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "fc1", "fc2", "gate_proj", "up_proj", "down_proj"]},
    #{"lora_rank": 32, "name": "LoRA-all", "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "fc1", "fc2", "gate_proj", "up_proj", "down_proj", "multi_modal_projector.linear"]}
]


results = []

# === Evaluate Zero-shot Baseline ===
wandb.init(project="DI725_Phase3", name="ZeroShot-Baseline")
zero_shot_results = evaluate_zero_shot(
    model_name="/kaggle/input/paligemma-2/transformers/paligemma2-3b-pt-224/1",
    image_dir=image_dir,
    val_df=val_df,
    num_samples=num_samples
)
wandb.log(zero_shot_results)
wandb.finish()

# Store Zero-shot results
results.append({
    "Model": "Zero-shot Baseline",
    "Rank": "N/A",
    "Modules": "N/A",
    "BLEU-4": zero_shot_results["zero_shot_BLEU"],
    "ROUGE-L": zero_shot_results["zero_shot_ROUGE_L"],
    "METEOR": zero_shot_results["zero_shot_METEOR"]
})

# === Train & Evaluate LoRA Configs ===
for config in configs:
    print(f"\n🚀 Training: {config['name']}")
    output_dir = f"./{config['name']}"

    wandb.init(
        project="DI725_Phase3",
        name=config["name"],
        config={
            "lora_rank": config.get("lora_rank"),
            "target_modules": "-".join(config.get("target_modules", ["q_proj", "v_proj"])),
            "num_samples_eval": num_samples
        },
        reinit=True
    )

    train_lora(
        model_name="/kaggle/input/paligemma-2/transformers/paligemma2-3b-pt-224/1",
        image_dir=image_dir,
        caption_file=caption_file,
        train_df=train_df,
        val_df=val_df,
        output_dir=output_dir,
        lora_rank=config.get("lora_rank", 32),
        target_modules=config.get("target_modules", ["q_proj", "v_proj"]),
        epochs=3,
        learning_rate=5e-4,
        batch_size=2,
        accum_steps=8
    )

    result = evaluate_lora_model(
        lora_model_path=output_dir,
        base_model_name="/kaggle/input/paligemma-2/transformers/paligemma2-3b-pt-224/1",
        image_dir=image_dir,
        val_df=val_df,
        num_samples=num_samples
    )

    # Add config info
    result["config"] = config["name"]
    result["rank"] = config.get("lora_rank", 32)
    result["modules"] = "-".join(config.get("target_modules", ["q_proj", "v_proj"]))

    # Log to WandB
    # wandb.log({
      #   "BLEU-4": result["lora_BLEU"],
        # "ROUGE-L": result["lora_ROUGE_L"],
         #"METEOR": result["lora_METEOR"]
     #})

    results.append({
        "Model": config["name"],
        "Rank": result["rank"],
        "Modules": result["modules"],
        "BLEU-4": result["lora_BLEU"],
        "ROUGE-L": result["lora_ROUGE_L"],
        "METEOR": result["lora_METEOR"]
    })

    wandb.finish()

# 📁 Save all results
results_df = pd.DataFrame(results)
results_df.to_csv("results_table.csv", index=False)
print("\n📊 Final Comparison Table:")
print(results_df.to_markdown(index=False))


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Running zero-shot evaluation...

Sample 0 - Hypothesis: caption
image result for image of container port in china
BLEU=0.025 ROUGE-L=0.000 METEOR=0.055

Sample 1 - Hypothesis: caption
invention can be seen through the window of invention .
BLEU=0.025 ROUGE-L=0.190 METEOR=0.100

Sample 2 - Hypothesis: caption
satellite image of the island
BLEU=0.081 ROUGE-L=0.211 METEOR=0.272

Sample 3 - Hypothesis: caption
property image # directly on the lake with private beach and pontoon
BLEU=0.040 ROUGE-L=0.143 METEOR=0.083

Sample 4 - Hypothesis: caption
police , ambulance and fire department vehicles line a street as seen in this aerial photo
BLEU=0.015 ROUGE-L=0.083 METEOR=0.081

Sample 5 - Hypothesis: caption
pedestrian bridge over the interstate
BLEU=0.035 ROUGE-L=0.000 METEOR=0.044

Sample 6 - Hypothesis: caption
aerial view of the runway .
BLEU=0.029 ROUGE-L=0.160 METEOR=0.088

Sample 7 - Hypothesis: caption
## th hole from the air .
BLEU=0.029 ROUGE-L=0.118 METEOR=0.064

Sample 8 - Hypothes

zero_shot_BLEU,▁
zero_shot_METEOR,▁
zero_shot_ROUGE_L,▁
zero_shot_BLEU,0.04368
zero_shot_METEOR,0.14035
zero_shot_ROUGE_L,0.11749


...

epoch,▁▁▅▅██
train_loss,█▁▁
val_loss,▁▆█
epoch,3
train_loss,11.96865
val_loss,11.98946


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Running LoRA model evaluation...


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



Sample 0 - Hypothesis: Describe this remote sensing instrument.
Describe the remote sensing instrument .
Describe the mobile sensing instrument .
BLEU=0.013 ROUGE-L=0.080 METEOR=0.052

Sample 1 - Hypothesis: What is shown in these satellite images?
what is what what what whatwhatwhatwhatwhatWhatWhatWhatWhat What What What What what what what WHATWHATWHATWHATWHAT WHAT WHAT WHAT WHATWHATWHAT WHATWHATWHATWhatWhatWhatwhatwhatwhat
BLEU=0.007 ROUGE-L=0.000 METEOR=0.056

Sample 2 - Hypothesis: Generate generate generate generate generategenerategenerategenerategenerate generate generate generate generation generation generation generation Generation Generation Generation Generation generation generation generation generate generate generate Generate generate generate generate generator generate generate generate generated generation generation generation GENERATION GENERATION GENERATION GENERATION GENERATE GENERATE GENERate generate generate
BLEU=0.000 ROUGE-L=0.000 METEOR=0.000

Sample 3 - 